In [1]:
# Cell 1 — Run FIRST after switching to T4 GPU
# Takes ~3 minutes

!pip install -q \
    langchain \
    langchain-community \
    sentence-transformers \
    faiss-cpu \
    PyMuPDF \
    pytesseract \
    Pillow \
    transformers \
    torch \
    gradio \
    accelerate \
    datasets \
    huggingface_hub

!apt-get install -q tesseract-ocr

print("All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
All packages installed!


In [2]:
# Cell 2 — Mount Drive (only for saving your FAISS index, ~50MB)
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/RAG_Project/outputs', exist_ok=True)

print("Drive mounted! Folder ready at MyDrive/RAG_Project/outputs")

Mounted at /content/drive
Drive mounted! Folder ready at MyDrive/RAG_Project/outputs


In [3]:
# Cell 3 — Load MyFixit dataset CORRECTLY from GitHub (direct JSON download)
# MyFixit is NOT on HuggingFace Hub — it's on GitHub as JSON files
# We download from the raw GitHub URL directly into /tmp (no Drive storage)
# Everyone gets EXACTLY the same 500 guides — no variation

import urllib.request
import json
import os

print("Loading MyFixit dataset from GitHub...")
print("(Downloads JSON directly to /tmp — no Drive storage used)\n")

# The MyFixit dataset is hosted on GitHub: rub-ksv/MyFixit-Dataset
# It has one JSON file per device category
# We use the Mac Laptop category — it's the annotated one with clean text

MYFIXIT_URLS = {
    "Mac_Laptop":   "https://raw.githubusercontent.com/rub-ksv/MyFixit-Dataset/master/Mac%20Laptop.json",
}

# Try the Mac Laptop file first (the annotated, clean one from the paper)
all_text = ""
count = 0
MAX_GUIDES = 500

for category, url in MYFIXIT_URLS.items():
    tmp_path = f"/tmp/myfixit_{category}.json"
    print(f"Downloading {category} JSON...")
    try:
        urllib.request.urlretrieve(url, tmp_path)
        with open(tmp_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        os.remove(tmp_path)  # delete immediately after reading
        print(f"  Downloaded  | {len(data)} guides found")

        for item in data:
            # Extract title + steps text from each guide
            title = item.get("Title", "Unknown")
            category_name = item.get("Category", "Unknown")
            steps = item.get("Steps", [])

            guide_text = f"\n\n=== GUIDE: {title} ===\n"
            guide_text += f"Category: {category_name}\n"
            for step in steps:
                lines = step.get("Lines", [])
                for line in lines:
                    guide_text += line.get("Text", "") + " "
            all_text += guide_text
            count += 1
            if count % 100 == 0:
                print(f"  Loaded {count} guides...")
            if count >= MAX_GUIDES:
                break

    except Exception as e:
        print(f"  Failed to download {category}: {e}")

print(f"\nMyFixit loaded! Guides: {count} | Characters: {len(all_text)}")
print(f"Preview:\n{all_text[:400]}")
# FALLBACK — if GitHub download fails, use iFixit public API directly
if count == 0:
    print("\nGitHub failed — using iFixit public API as fallback...")
    import time

    GUIDE_IDS = list(range(816, 1316))  # 500 guide IDs starting from guide 816
    for guide_id in GUIDE_IDS:
        try:
            api_url = f"https://www.ifixit.com/api/2.0/guides/{guide_id}"
            with urllib.request.urlopen(api_url, timeout=5) as resp:
                guide = json.loads(resp.read())
            title = guide.get("title", "Unknown")
            steps = guide.get("steps", [])
            guide_text = f"\n\n=== GUIDE: {title} ===\n"
            for step in steps:
                for line in step.get("lines", []):
                    guide_text += line.get("text", "") + " "
            all_text += guide_text
            count += 1
            if count % 50 == 0:
                print(f"  Loaded {count} guides via API...")
                time.sleep(1)  # be polite to the API
            if count >= MAX_GUIDES:
                break
        except:
            continue  # skip guides that 404 or timeout

    print(f"\nAPI fallback done! Guides: {count} | Characters: {len(all_text)}")

Loading MyFixit dataset from GitHub...
(Downloads JSON directly to /tmp — no Drive storage used)

  Failed to download Mac_Laptop: HTTP Error 404: Not Found

MyFixit loaded! Guides: 0 | Characters: 0
Preview:


GitHub failed — using iFixit public API as fallback...
  Loaded 50 guides via API...
  Loaded 100 guides via API...
  Loaded 150 guides via API...
  Loaded 200 guides via API...
  Loaded 250 guides via API...
  Loaded 300 guides via API...
  Loaded 350 guides via API...
  Loaded 400 guides via API...

API fallback done! Guides: 429 | Characters: 45762


In [4]:
# Cell 4 — Load Sony BA-4 + Samsung CRT from Internet Archive
# Using the FULL TEXT (.txt) files instead of PDFs
# Internet Archive always hosts these — no 503 errors, no OCR needed
# Plain text = cleaner than PDF extraction anyway

import urllib.request

print("Loading pilot manuals from Internet Archive (plain text format)...\n")

PILOT_MANUALS = {
    "sony_ba4": "https://archive.org/download/kv13m_series/kv13m_series_djvu.txt",

    "samsung_crt_s51a": "https://archive.org/stream/manual_CK22B63V3X_SM_SAMSUNG_EN/CK22B63V3X_SM_SAMSUNG_EN_djvu.txt",

    "samsung_crt_70gs": "https://archive.org/stream/manualzz-id-873404/873404_djvu.txt",
}

pdf_text = ""

for label, url in PILOT_MANUALS.items():
    print(f"Downloading {label} (plain text)...")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            text = resp.read().decode("utf-8", errors="ignore")
        pdf_text += f"\n\n=== SOURCE: {label} ===\n{text}"
        print(f"   Done — {len(text)} characters")
    except Exception as e:
        print(f"   Failed: {e}")

print(f"\nPilot manuals loaded! Total characters: {len(pdf_text)}")
print(f"Preview:\n{pdf_text[:400]}")

Loading pilot manuals from Internet Archive (plain text format)...

   Done — 282524 characters
   Done — 298404 characters
   Done — 321525 characters

Pilot manuals loaded! Total characters: 902550
Preview:


=== SOURCE: sony_ba4 ===
— 1 — 


KV-[13M40/50/51], KV-[14MB40/40C/40A], KV-[20M40/S40/S41/V80], KV-[21SE40/40A/40C/80/80A/80C], 
KV-[21MB40C/40M/40P], KV-[21ME40/40P], KV-[21SB40/40M/40P], KV-21XT4A 


TRINITRON® COLOR TV 


KV-13M40 
RM-Y156 
US 
SCC-S01E-A 
KV-13M40 
RM-Y156 
CND 
SCC-S03D-A 
KV-13M50 
RM-Y156 
US 
SCC-S01G-A 
KV-13M51 
RM-Y156W 
US 
SCC-S01F-A 
KV-14MB40 
RM-Y156 
E 
SCC-S04


In [5]:
# Cell 5 — Merge MyFixit dataset + pilot PDFs, then clean

import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)              # collapse whitespace
    text = re.sub(r'page \d+', '', text, flags=re.IGNORECASE)  # remove "page 12"
    text = re.sub(r'\.{3,}', '', text)            # remove "......"
    text = re.sub(r'[^\w\s\.\,\:\;\-\(\)]', '', text)  # remove junk chars
    return text.strip()

# Combine everything
combined_raw = all_text + pdf_text
cleaned_text = clean_text(combined_raw)

print(f"Combined knowledge base ready!")
print(f"   Raw characters    : {len(combined_raw)}")
print(f"   Cleaned characters: {len(cleaned_text)}")
print(f"\nPreview:\n{cleaned_text[:400]}")

Combined knowledge base ready!
   Raw characters    : 948312
   Cleaned characters: 751522

Preview:
GUIDE: MacBook Unibody Model A1278 Hard Drive Replacement   GUIDE: iPhone 3GS Teardown   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Access Door Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Battery Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Hard Drive Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Lower Case Replace


In [6]:
# Cell 6 — Split text into 512-token chunks with 64-token overlap
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ← fixed import

def chunk_text(text, chunk_size=512, chunk_overlap=64):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "]
    )
    return splitter.split_text(text)

chunks = chunk_text(cleaned_text)

print(f"Chunking complete!")
print(f"   Total chunks: {len(chunks)}")
print(f"\nSample chunk 1:\n{chunks[0]}")
print(f"\nSample chunk 2:\n{chunks[1]}")

Chunking complete!
   Total chunks: 1873

Sample chunk 1:
GUIDE: MacBook Unibody Model A1278 Hard Drive Replacement   GUIDE: iPhone 3GS Teardown   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Access Door Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Battery Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Hard Drive Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Lower Case Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 RAM Replacement   GUIDE: MacBook Pro 15 Unibody

Sample chunk 2:
and Early 2009 RAM Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Optical Drive Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Hard Drive Cable Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Mid Wall Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 2009 Optical Drive Replacement   GUIDE: MacBook Pro 15 Unibody Late 2008 and Early 20

In [7]:
# Cell 7 — Load intfloat/multilingual-e5-small
# Supports 100+ languages including Kannada, Hindi, Tamil
# Takes ~2 mins to download first time

from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading multilingual embedding model...")
print("(Downloads ~150MB — once per session)\n")

embedding_model = SentenceTransformer('intfloat/multilingual-e5-small')

print("Embedding model loaded!")
print(f"   Model: intfloat/multilingual-e5-small")
print(f"   Embedding dim: 384")
print(f"   Supported languages: 100+")

def embed_passages(chunks):
    """Embed document chunks. e5 models REQUIRE 'passage: ' prefix."""
    passages = [f"passage: {chunk}" for chunk in chunks]
    embeddings = embedding_model.encode(
        passages,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True  # required for cosine similarity via dot product
    )
    return embeddings

def embed_query(query):
    """Embed a user query. e5 models REQUIRE 'query: ' prefix."""
    return embedding_model.encode(
        f"query: {query}",
        normalize_embeddings=True
    )

Loading multilingual embedding model...
(Downloads ~150MB — once per session)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding model loaded!
   Model: intfloat/multilingual-e5-small
   Embedding dim: 384
   Supported languages: 100+


In [8]:
# Cell 8 — Build FAISS index and save to Drive
# Run this ONCE. Next session just click "Load Saved Index" in the UI.

import faiss
import pickle

print("Embedding all chunks... (takes ~45 secs on T4 GPU)")
embeddings = embed_passages(chunks)
print(f"\nEmbeddings shape: {embeddings.shape}")

# Build FAISS index (IndexFlatIP = exact cosine search on normalized vectors)
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype('float32'))
print(f"FAISS index built — {index.ntotal} vectors, dim={dimension}")

# Save to Drive
faiss.write_index(index, '/content/drive/MyDrive/RAG_Project/outputs/manual_index.faiss')
with open('/content/drive/MyDrive/RAG_Project/outputs/chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

print("\nSaved to Drive!")
print("   /RAG_Project/outputs/manual_index.faiss")
print("   /RAG_Project/outputs/chunks.pkl")
print("\nNext session: skip this cell, use 'Load Saved Index' in the UI.")

Embedding all chunks... (takes ~45 secs on T4 GPU)


Batches:   0%|          | 0/59 [00:00<?, ?it/s]


Embeddings shape: (1873, 384)
FAISS index built — 1873 vectors, dim=384

Saved to Drive!
   /RAG_Project/outputs/manual_index.faiss
   /RAG_Project/outputs/chunks.pkl

Next session: skip this cell, use 'Load Saved Index' in the UI.


In [9]:
# Cell 9 — Load flan-t5-base (direct model loading, more stable than pipeline)
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

model_name = "google/flan-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name} on {device}...")

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32  # ← float32 on CPU, float16 only on GPU
).to(device)

model.eval()  # set to inference mode

def generator(prompt, max_new_tokens=200, do_sample=False):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512   # truncate long prompts to fit T5's context window
    ).to(model.device)
    with torch.no_grad():  # saves memory during inference
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample)
    return [{"generated_text": tokenizer.decode(outputs[0], skip_special_tokens=True)}]

print(f"LLM loaded! | Device: {device.upper()} | Model: {model_name}")

Loading google/flan-t5-base on cuda...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded! | Device: CUDA | Model: google/flan-t5-base


In [10]:
# Cell 10 — RAG Pipeline (FIXED: Strict Context Guardrails)
!pip install -q deep-translator

from deep_translator import GoogleTranslator
import numpy as np

LANG_CODES = {
    "English": "en",
    "Kannada": "kn",
    "Hindi":   "hi",
    "Tamil":   "ta",
}

# ── Retrieve ──────────────────────────────────────────────────────────────────
def retrieve_top_k(query, faiss_index, text_chunks, k=5):
    query_emb = embed_query(query)
    query_emb = np.array([query_emb]).astype('float32')
    distances, indices = faiss_index.search(query_emb, k)
    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        results.append({
            "rank":  rank + 1,
            "chunk": text_chunks[idx],
            "score": float(dist)
        })
    return results

# ── Generate (With Strict Guardrails to prevent random answers) ────────
def generate_answer(query, retrieved_chunks):
    if not retrieved_chunks:
        return "I'm sorry, but the selected manuals do not contain information regarding this topic."

    # Use a bit more context for better checking
    context = "\n".join([
        f"[Source {r['rank']}]: {r['chunk'].strip()}"
        for r in retrieved_chunks[:3]
    ])

    # Modified prompt with strict negative constraints
    prompt = (
        "You are a professional repair assistant. Use ONLY the provided context to answer the question.\n"
        "STRICT RULES:\n"
        "1. If the context is unrelated to the question (e.g., context is about software/coding but question is about TV repair), say: 'I'm sorry, but the selected manuals do not contain information regarding this topic.'\n"
        "2. Do not use any outside knowledge.\n"
        "3. If the answer is not explicitly in the context, do not make it up.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Step-by-step answer:"
    )

    try:
        # Note: generator needs to be defined in your previous cells
        output = generator(prompt, max_new_tokens=200)

        if isinstance(output, list) and len(output) > 0:
            answer = output[0].get('generated_text', '').strip()
        else:
            answer = str(output)

        # Final safety check: if the model just repeats the prompt or gives a very short unrelated response
        if not answer or len(answer) < 5:
            return "I'm sorry, but the selected manuals do not contain information regarding this topic."

        return answer

    except Exception as e:
        print(f"[Generation Error]: {e}")
        return "I'm sorry, but an error occurred during answer generation."

# ── Translate any text to target language ─────────────────────────────────────
def translate_text(text, target_language):
    """Translate text to target language. Handles long text by chunking."""
    if not text:
        return text

    lang_code = LANG_CODES.get(target_language, "en")
    if lang_code == "en":
        return text   # no translation needed

    try:
        # Split into ≤4500-char pieces (Google Translate limit)
        parts = [text[i:i+4500] for i in range(0, len(text), 4500)]
        translated = []
        for part in parts:
            result = GoogleTranslator(source='en', target=lang_code).translate(part)
            translated.append(result)
        return "\n".join(translated)
    except Exception as e:
        print(f"[Translation failed: {e}]")
        return text   # fallback: return original

# ── Format source citations ────────────────────────────────────────────────────
def format_citations(retrieved_chunks, target_language="English"):
    """
    Returns source chunks translated into target_language.
    Only the snippet preview is translated — score stays in English (numeric).
    """
    lines = []
    for r in retrieved_chunks[:3]:
        snippet = r['chunk'][:300].strip()
        translated_snippet = translate_text(snippet, target_language)
        lines.append(
            f"▸ [Source {r['rank']}]  Score: {r['score']:.3f}\n"
            f"{translated_snippet}..."
        )
    return "\n\n".join(lines)

# ── Full RAG pipeline ─────────────────────────────────────────────────────────
def rag_answer(query, faiss_index, text_chunks, target_language="English"):
    # 1. Retrieve
    retrieved = retrieve_top_k(query, faiss_index, text_chunks, k=5)

    # 2. Generate answer in English
    answer_english = generate_answer(query, retrieved)

    # 3. Translate the FULL answer to target language
    answer_translated = translate_text(answer_english, target_language)

    # 4. Format citations — also translated
    citations = format_citations(retrieved, target_language)

    return answer_translated, citations

print("✅ RAG pipeline ready with strict context guardrails!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.7 MB/s eta 0:00:00
✅ RAG pipeline ready with strict context guardrails!


PERFECT!

In [11]:
# Cell 12 — Gradio UI
# Run this cell last

import gradio as gr
import faiss, pickle, fitz, re, os
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter

current_index  = index
current_chunks = chunks

uploaded_manual_registry = {}
BASE_CHUNK_COUNT = len(chunks)

RELEVANCE_THRESHOLD = 0.50   # chunks scoring below this are treated as "not found"

# ── PDF text extraction ───────────────────────────────────────────────────────
def load_pdf_text_from_file(path):
    doc = fitz.open(path)
    full_text = ""
    for page_num, page in enumerate(doc):
        text = page.get_text("text")
        if len(text.strip()) < 50:
            try:
                import pytesseract
                from PIL import Image
                pix = page.get_pixmap(dpi=200)
                img_path = f"/tmp/ui_page_{page_num}.png"
                pix.save(img_path)
                text = pytesseract.image_to_string(Image.open(img_path))
            except Exception:
                pass
        full_text += f"\n{text}"
    doc.close()
    text = re.sub(r'https?://\S+', '', full_text)
    text = re.sub(r'\b(href|class|id|src|div|span|svg|nav|aria|xmlns|style|data-[a-z\-]+)\S*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'--lit[a-zA-Z0-9\-]+--', '', text)
    lines = text.split('\n')
    clean_lines = []
    for line in lines:
        stripped = line.strip()
        if len(stripped) < 3:
            continue
        alpha_ratio = sum(c.isalpha() or c.isspace() for c in stripped) / max(len(stripped), 1)
        if alpha_ratio > 0.4:
            clean_lines.append(stripped)
    return "\n".join(clean_lines)

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'page \d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\.{3,}', '', text)
    text = re.sub(r'[^\w\s\.\,\:\;\-\(\)]', '', text)
    return text.strip()

# ── Index management ──────────────────────────────────────────────────────────
def load_saved_index():
    global current_index, current_chunks
    try:
        current_index = faiss.read_index('/content/drive/MyDrive/RAG_Project/outputs/manual_index.faiss')
        with open('/content/drive/MyDrive/RAG_Project/outputs/chunks.pkl', 'rb') as f:
            current_chunks = pickle.load(f)
        return f"Index loaded — {len(current_chunks)} chunks ready!"
    except Exception as e:
        return f"Error: {e}"

# ── Multi-file upload ─────────────────────────────────────────────────────────
def add_user_manual(pdf_files):
    global current_index, current_chunks, uploaded_manual_registry
    if not pdf_files:
        return "No files uploaded.", gr.Dropdown(choices=get_scope_choices(), value=[get_scope_choices()[0]], multiselect=True)

    results = []
    for pdf_file in pdf_files:
        try:
            raw = load_pdf_text_from_file(pdf_file.name)
            preview = raw[:300].replace('\n', ' ')
            print(f"[DEBUG] {pdf_file.name} preview:\n{preview}\n")

            if len(raw.strip()) < 100:
                results.append(f"⚠️ {os.path.basename(pdf_file.name)}: too short or extraction failed.")
                continue

            clean = clean_text(raw)
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=512, chunk_overlap=64,
                separators=["\n\n", "\n", ".", " "]
            )
            new_chunks = splitter.split_text(clean)
            new_embs   = embed_passages(new_chunks)

            start_idx = len(current_chunks)
            current_index.add(new_embs.astype('float32'))
            current_chunks.extend(new_chunks)
            end_idx = len(current_chunks)

            fname = os.path.basename(pdf_file.name)
            uploaded_manual_registry[fname] = list(range(start_idx, end_idx))

            faiss.write_index(current_index, '/content/drive/MyDrive/RAG_Project/outputs/manual_index.faiss')
            with open('/content/drive/MyDrive/RAG_Project/outputs/chunks.pkl', 'wb') as f:
                pickle.dump(current_chunks, f)

            results.append(
                f"✅ {fname}: {len(new_chunks)} chunks added\n"
                f"   Preview: {raw[:200].replace(chr(10), ' ')}..."
            )
        except Exception as e:
            results.append(f"❌ {os.path.basename(pdf_file.name)}: Error — {e}")

    new_choices = get_scope_choices()
    return "\n\n".join(results), gr.Dropdown(choices=new_choices, value=new_choices, multiselect=True)

# ── Scope helpers ─────────────────────────────────────────────────────────────
def get_scope_choices():
    choices = ["🌐 All Manuals", "📦 Base Only (Sony + Samsung + MyFixit)"]
    for fname in uploaded_manual_registry:
        choices.append(f"📄 {fname}")
    return choices

# ── Retrieval with threshold ──────────────────────────────────────────────────
def retrieve_scoped(query, scope_labels, k=5):
    """
    Retrieve top-k chunks from selected scopes.
    Returns only chunks that pass RELEVANCE_THRESHOLD.
    scope_labels is a list (multiselect).
    """
    if not scope_labels:
        return []
    if isinstance(scope_labels, str):
        scope_labels = [scope_labels]

    query_emb = embed_query(query).astype('float32')

    # ── If "All Manuals" is among selections → use full FAISS index ──────────
    if any(label.startswith("🌐") for label in scope_labels):
        q = query_emb.reshape(1, -1)
        distances, indices = current_index.search(q, k)
        results = []
        for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            if idx >= 0 and float(dist) >= RELEVANCE_THRESHOLD:
                results.append({"rank": rank+1, "chunk": current_chunks[idx], "score": float(dist)})
        return results

    # ── Build pool from all selected specific sources ─────────────────────────
    combined_indices = []
    for label in scope_labels:
        if label.startswith("📦"):
            combined_indices.extend(list(range(BASE_CHUNK_COUNT)))
        elif label.startswith("📄"):
            fname = label.replace("📄 ", "")
            if fname in uploaded_manual_registry:
                combined_indices.extend(uploaded_manual_registry[fname])

    unique_indices = list(dict.fromkeys(combined_indices))
    scoped_chunks  = [current_chunks[i] for i in unique_indices]

    if not scoped_chunks:
        return []

    # Brute-force cosine similarity over the pool
    scoped_embs = embedding_model.encode(
        [f"passage: {c}" for c in scoped_chunks],
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype('float32')

    scores    = np.dot(scoped_embs, query_emb)
    top_k_idx = np.argsort(scores)[::-1][:k]

    # ── Apply threshold — only return genuinely relevant chunks ──────────────
    results = []
    for rank, idx in enumerate(top_k_idx):
        if float(scores[idx]) >= RELEVANCE_THRESHOLD:
            results.append({
                "rank":  rank + 1,
                "chunk": scoped_chunks[idx],
                "score": float(scores[idx])
            })
    return results

# ── Ask question ──────────────────────────────────────────────────────────────
def ask_question(query, language, scope_labels):
    if not query.strip():
        return "Please enter a question first.", ""
    if not scope_labels:
        return "⚠️ Please select at least one source in the Search Scope dropdown.", ""
    try:
        retrieved = retrieve_scoped(query, scope_labels, k=5)

        # Nothing passed the threshold → topic not in selected manuals
        if not retrieved:
            not_found_msg = "I'm sorry, but the selected manuals do not contain information regarding this topic."
            return translate_text(not_found_msg, language), "No relevant source chunks found in the selected manuals."

        answer_en  = generate_answer(query, retrieved)
        answer_out = translate_text(answer_en, language)
        citations  = format_citations(retrieved, language)
        return answer_out, citations
    except Exception as e:
        return f"❌ Error: {e}", ""

def show_status():
    uploaded_info = "\n".join(
        f"   📄 {fname}: {len(idxs)} chunks"
        for fname, idxs in uploaded_manual_registry.items()
    ) or "   None yet"
    return (f"Knowledge Base       →  {len(current_chunks)} chunks\n"
            f"Base chunks          →  {BASE_CHUNK_COUNT}\n"
            f"FAISS Index          →  {current_index.ntotal} vectors  |  dim 384\n"
            f"Embedding Model      →  intfloat/multilingual-e5-small\n"
            f"LLM                  →  google/flan-t5-base\n"
            f"Languages Supported  →  Kannada · Hindi · Tamil · English · +96 more\n"
            f"Pilot Manuals        →  Sony BA-4 · Samsung CRT S51A · Samsung CRT 70GS\n"
            f"Dataset              →  MyFixit (500 guides streamed)\n"
            f"Uploaded Manuals:\n{uploaded_info}")

# ── CSS ───────────────────────────────────────────────────────────────────────
css = """
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=DM+Sans:ital,wght@0,300;0,400;0,500;1,300&display=swap');
:root {
  --bg:#0a0a0f; --surface:#13131a; --surface2:#1c1c27; --border:#2a2a3d;
  --accent:#7c6af7; --accent2:#f76a8a; --accent3:#6af7c8;
  --text:#e8e8f0; --muted:#7a7a9a; --glow:rgba(124,106,247,0.25);
}
* { box-sizing: border-box; }
body, .gradio-container {
  background: var(--bg) !important; font-family: 'DM Sans', sans-serif !important;
  color: var(--text) !important; min-height: 100vh;
}
.gradio-container::before {
  content:''; position:fixed; inset:0;
  background-image: linear-gradient(rgba(124,106,247,0.03) 1px,transparent 1px),
                    linear-gradient(90deg,rgba(124,106,247,0.03) 1px,transparent 1px);
  background-size:40px 40px; pointer-events:none; z-index:0;
}
.hero { text-align:center; padding:48px 24px 32px; position:relative; }
.hero-badge {
  display:inline-block;
  background:linear-gradient(135deg,rgba(124,106,247,0.15),rgba(247,106,138,0.15));
  border:1px solid var(--border); border-radius:100px; padding:6px 18px;
  font-size:11px; font-weight:600; letter-spacing:2px; text-transform:uppercase;
  color:var(--accent); margin-bottom:20px;
}
.hero h1 {
  font-family:'Syne',sans-serif !important; font-size:clamp(28px,5vw,52px) !important;
  font-weight:800 !important; line-height:1.1 !important;
  background:linear-gradient(135deg,#fff 0%,var(--accent) 50%,var(--accent2) 100%);
  -webkit-background-clip:text; -webkit-text-fill-color:transparent; background-clip:text;
  margin:0 0 12px !important; padding:0 !important;
}
.hero p { color:var(--muted); font-size:15px; font-weight:300; margin:0; letter-spacing:0.3px; }
.hero-pills { display:flex; flex-wrap:wrap; justify-content:center; gap:8px; margin-top:20px; }
.pill { background:var(--surface); border:1px solid var(--border); border-radius:100px; padding:5px 14px; font-size:12px; color:var(--muted); }
.pill span { color:var(--accent3); font-weight:500; }
.tab-nav { border-bottom:1px solid var(--border) !important; background:transparent !important; }
.tab-nav button {
  font-family:'Syne',sans-serif !important; font-size:13px !important; font-weight:600 !important;
  letter-spacing:0.5px !important; color:var(--muted) !important; background:transparent !important;
  border:none !important; padding:12px 20px !important; transition:color 0.2s !important;
}
.tab-nav button.selected { color:var(--accent) !important; border-bottom:2px solid var(--accent) !important; }
.tab-nav button:hover { color:var(--text) !important; }
.card {
  background:var(--surface); border:1px solid var(--border); border-radius:16px;
  padding:24px; margin-bottom:16px; position:relative; overflow:hidden;
}
.card::before {
  content:''; position:absolute; top:0; left:0; right:0; height:1px;
  background:linear-gradient(90deg,transparent,var(--accent),transparent); opacity:0.5;
}
.card-title { font-family:'Syne',sans-serif; font-size:13px; font-weight:700; letter-spacing:1.5px; text-transform:uppercase; color:var(--accent); margin-bottom:8px; }
.card-desc { font-size:13px; color:var(--muted); line-height:1.6; margin-bottom:16px; }
label { font-size:12px !important; font-weight:600 !important; letter-spacing:1px !important; text-transform:uppercase !important; color:var(--muted) !important; margin-bottom:6px !important; }
textarea, input[type=text], .gr-input {
  background:var(--surface2) !important; border:1px solid var(--border) !important;
  border-radius:10px !important; color:var(--text) !important; font-family:'DM Sans',sans-serif !important;
  font-size:14px !important; padding:12px 14px !important; transition:border-color 0.2s !important;
}
textarea:focus, input:focus { border-color:var(--accent) !important; box-shadow:0 0 0 3px var(--glow) !important; outline:none !important; }
.gr-dropdown, select { background:var(--surface2) !important; border:1px solid var(--border) !important; border-radius:10px !important; color:var(--text) !important; font-family:'DM Sans',sans-serif !important; }
button.primary, .gr-button-primary {
  background:linear-gradient(135deg,var(--accent),#9b5de5) !important;
  border:none !important; border-radius:10px !important; color:#fff !important;
  font-family:'Syne',sans-serif !important; font-weight:700 !important; font-size:13px !important;
  letter-spacing:0.5px !important; padding:12px 24px !important; cursor:pointer !important;
  transition:opacity 0.2s,transform 0.15s !important; box-shadow:0 4px 20px var(--glow) !important;
}
button.primary:hover { opacity:0.88 !important; transform:translateY(-1px) !important; }
button.secondary, .gr-button-secondary {
  background:var(--surface2) !important; border:1px solid var(--border) !important;
  border-radius:10px !important; color:var(--text) !important; font-family:'Syne',sans-serif !important;
  font-weight:600 !important; font-size:13px !important; padding:10px 20px !important; transition:border-color 0.2s !important;
}
button.secondary:hover { border-color:var(--accent) !important; }
.output-box { background:var(--surface2) !important; border:1px solid var(--border) !important; border-radius:12px !important; padding:16px !important; font-size:14px !important; line-height:1.7 !important; color:var(--text) !important; font-family:'DM Sans',sans-serif !important; }
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.4} }
.stats-row { display:flex; gap:12px; flex-wrap:wrap; margin-bottom:16px; }
.stat-card { flex:1; min-width:120px; background:var(--surface2); border:1px solid var(--border); border-radius:12px; padding:16px; text-align:center; }
.stat-number { font-family:'Syne',sans-serif; font-size:22px; font-weight:800; background:linear-gradient(135deg,var(--accent),var(--accent2)); -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.stat-label { font-size:11px; color:var(--muted); letter-spacing:1px; text-transform:uppercase; margin-top:4px; }
.gr-file { background:var(--surface2) !important; border:2px dashed var(--border) !important; border-radius:12px !important; transition:border-color 0.2s !important; }
.gr-file:hover { border-color:var(--accent) !important; }
::-webkit-scrollbar { width:6px; }
::-webkit-scrollbar-track { background:var(--bg); }
::-webkit-scrollbar-thumb { background:var(--border); border-radius:3px; }
::-webkit-scrollbar-thumb:hover { background:var(--accent); }
"""

# ── Build UI ──────────────────────────────────────────────────────────────────
with gr.Blocks(css=css, title="FixBot — Multilingual RAG") as demo:

    gr.HTML("""
    <div class="hero">
      <div class="hero-badge">⚡ Multilingual RAG System</div>
      <h1>FixBot</h1>
      <p>AI-powered repair assistant for consumer electronics — ask in any language</p>
      <div class="hero-pills">
        <div class="pill"><span>🔧</span> Sony BA-4</div>
        <div class="pill"><span>📺</span> Samsung CRT</div>
        <div class="pill"><span>📚</span> MyFixit 500 Guides</div>
        <div class="pill"><span>🌐</span> 100+ Languages</div>
        <div class="pill"><span>✅</span> Scoped Search</div>
      </div>
    </div>
    """)

    with gr.Tabs(elem_classes="tab-nav"):

        # ── TAB 1: SYSTEM ─────────────────────────────────────────────────────
        with gr.Tab("System"):
            gr.HTML("""
            <div class="card">
              <div class="card-title">Knowledge Base</div>
              <div class="stats-row">
                <div class="stat-card"><div class="stat-number">500</div><div class="stat-label">MyFixit Guides</div></div>
                <div class="stat-card"><div class="stat-number">3</div><div class="stat-label">Pilot Manuals</div></div>
                <div class="stat-card"><div class="stat-number">100+</div><div class="stat-label">Languages</div></div>
                <div class="stat-card"><div class="stat-number">384</div><div class="stat-label">Embedding Dim</div></div>
              </div>
            </div>
            """)
            with gr.Row():
                with gr.Column(scale=1):
                    status_btn = gr.Button("Check System Status", variant="secondary")
                with gr.Column(scale=1):
                    load_btn = gr.Button("Load Saved Index", variant="secondary")
            status_out = gr.Textbox(label="System Info", lines=10, elem_classes="output-box")
            load_out   = gr.Textbox(label="Load Status",  lines=2,  elem_classes="output-box")
            status_btn.click(show_status,    outputs=[status_out])
            load_btn.click(load_saved_index, outputs=[load_out])

        # ── TAB 2: UPLOAD MANUAL ──────────────────────────────────────────────
        with gr.Tab("Upload Manual"):
            gr.HTML("""
            <div class="card">
              <div class="card-title">Add Your Own Manuals</div>
              <div class="card-desc">
                Upload one or more PDF manuals. Each file is tracked separately —
                select any combination in Ask FixBot to scope your answers precisely.
              </div>
            </div>
            """)
            pdf_input  = gr.File(
                label="Drop PDFs here (select multiple files at once)",
                file_types=[".pdf"],
                file_count="multiple"
            )
            upload_btn = gr.Button("Add to Knowledge Base", variant="primary")
            upload_out = gr.Textbox(label="Upload Status", lines=8, elem_classes="output-box")

        # ── TAB 3: ASK ────────────────────────────────────────────────────────
        with gr.Tab("Ask FixBot"):
            gr.HTML("""
            <div class="card">
              <div class="card-title">Ask a Repair Question</div>
              <div class="card-desc">
                Select one or more sources below. Answers only come from what you tick —
                if the topic isn't there, you'll be told clearly. Type in any language.
              </div>
            </div>
            """)

            with gr.Row():
                with gr.Column(scale=3):
                    query_in = gr.Textbox(
                        label="Your Question",
                        placeholder="e.g.  How to replace a capacitor in a CRT TV?  /  ಟಿವಿ ರಿಪೇರಿ ಹೇಗೆ?",
                        lines=3
                    )
                with gr.Column(scale=1):
                    lang_in = gr.Dropdown(
                        choices=["English", "Kannada", "Hindi", "Tamil"],
                        label="Answer Language",
                        value="English"
                    )

            # ── Multi-select scope dropdown ───────────────────────────────────
            init_choices = get_scope_choices()
            scope_in = gr.Dropdown(
                choices=init_choices,
                value=[init_choices[0]],      # default: All Manuals selected
                multiselect=True,
                label="🔍 Search Scope  —  select one or more manuals (answers only come from ticked sources)"
            )

            ask_btn = gr.Button("Get Answer", variant="primary")
            ans_out = gr.Textbox(label="Generated Answer",        lines=8,  elem_classes="output-box")
            src_out = gr.Textbox(label="Retrieved Source Chunks", lines=10, elem_classes="output-box")

            ask_btn.click(
                ask_question,
                inputs=[query_in, lang_in, scope_in],
                outputs=[ans_out, src_out]
            )

    # ── FOOTER ────────────────────────────────────────────────────────────────
    gr.HTML("""
    <div style="text-align:center; padding:32px 0 16px; color:#3a3a5a; font-size:12px; letter-spacing:1px;">
      FIXBOT &nbsp;·&nbsp; MULTILINGUAL RAG &nbsp;·&nbsp; TEAM 5 &nbsp;·&nbsp; ANJALI · AVANI · NEEHARIKA
    </div>
    """)

    # ── Wire upload AFTER all widgets defined so scope_in exists ─────────────
    def handle_upload(pdf_files):
        status_text, _ = add_user_manual(pdf_files)
        new_choices    = get_scope_choices()
        return (
            status_text,
            gr.Dropdown(choices=new_choices, value=[new_choices[0]], multiselect=True)
        )

    upload_btn.click(
        handle_upload,
        inputs=[pdf_input],
        outputs=[upload_out, scope_in]   # directly refreshes scope_in in Ask tab
    )

demo.launch(share=True)
print("\nFixBot UI launched! Open the URL above.")

/tmp/ipykernel_2749/2479421303.py:311: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="FixBot — Multilingual RAG") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7e4d35f3610b75a56d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



FixBot UI launched! Open the URL above.
